# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Record sets and fields are referenced by `@id` as per Croissant specification.

In [ ]:
# List all record sets (by @id) and their fields
record_sets = list(metadata.record_sets)
print("Record sets in the dataset (by @id and name):")
for rs in record_sets:
    print(f"- @id: {rs.id}\tName: {rs.name}")

# For illustration, display fields and columns for each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    for field in rs.fields:
        print(f"  Field: {field.name} (@id: {field.id}) dataType: {field.data_type}")
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"    Column: {col.name} (@id: {col.id})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose record set(s) to load by @id (replace with actual IDs from the previous overview)
record_set_ids = [rs.id for rs in metadata.record_sets]  # List all record set IDs

dfs = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(records)
    print(f"  Columns: {dfs[record_set_id].columns.tolist()}")
    print(dfs[record_set_id].head(), "\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. All fields referenced by their `@id`.

In [ ]:
# Example EDA using the first record set and a numeric field
if record_set_ids:
    # For demonstration, pick the first record set
    example_record_set_id = record_set_ids[0]
    df = dfs[example_record_set_id]

    # Show columns and try to infer a numeric field (by @id, but fallback to name)
    print(f"Fields in {example_record_set_id}: {df.columns.tolist()}")
    # Heuristically choose a relevant numeric field
    possible_numeric_fields = [c for c in df.columns if df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[c])]
    if not possible_numeric_fields:
        # Try to coerce numeric columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        possible_numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]  # Using the column name (should match field @id)
        print(f"Chosen numeric field: {numeric_field_id}")

        # Apply a threshold filter
        threshold = df[numeric_field_id].mean()  # Use mean as an example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (Total: {len(filtered_df)} records):")
        print(filtered_df.head())

        # Normalize the chosen numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categoric field (search for 'description', 'type', or first object field)
        group_field = None
        candidate_group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field_id]
        if candidate_group_fields:
            group_field = candidate_group_fields[0]

        if group_field:
            print(f"\nGrouping by field: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(grouped.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric fields found in the data to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll use matplotlib/seaborn for visualizations. Adjust field `@id` and visualization as appropriate for the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and possible_numeric_fields:
    # Histogram of the main numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {example_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a group field exists, make a boxplot
    if group_field is not None:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we were able to load metadata, record sets, and perform basic processing on the Mass Spectrometry EV dataset.
- Numeric field(s) (referenced by their `@id`) allowed for filtering, normalization, and visualization.
- Group-based exploration hints at relationships between categorical and numeric variables.
- For complete understanding, consult the Croissant metadata and referenced field/record set `@id`s in the schema.

---

_For additional analysis, extend the notebook by referencing more record sets, fields, or by integrating custom analysis._